# Deploy RAG Agent (dev)
Este notebook **no genera** `rag_agent.py`. Asume que ya existe en:

`/Workspace/Users/<tu_usuario>/bind_agent/rag/rag_agent.py`

Objetivo:
1) Log + register del modelo en **Unity Catalog** (MLflow registry).
2) Crear/actualizar un **Model Serving endpoint** que sirva ese modelo.
3) Probar una invocación simple.

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# ==============================
# 1) Dependencias (solo para pruebas locales)
# ==============================
# En Serverless compute puede no venir mlflow instalado.
%pip install -U databricks-vectorsearch mlflow databricks-sdk
dbutils.library.restartPython()


In [0]:
# ==============================
# 2) Config (ajustar a tu workspace)
# ==============================
from pathlib import Path
import time

# --- Vector Search ---
VS_ENDPOINT = "bind_agent_vs"
VS_INDEX_FULL_NAME = "bind_agent.docs.pdf_chunks_vs_idx"

# --- Endpoints ---
EMBED_ENDPOINT = "databricks-bge-large-en"  # for query_vector
EMB_ENDPOINT = EMBED_ENDPOINT  # backward-compatible alias
LLM_ENDPOINT = "databricks-gemma-3-12b"
# LLM_ENDPOINT = _get_env("RAG_LLM_ENDPOINT", default="databricks-gpt-5-2")

# --- Retrieval params ---
TOP_K_CANDIDATES = "40"
TOP_K_FINAL = "8"
LEX_FALLBACK_LIMIT = "20"

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS = "14000"
RERANK_SNIPPET_CHARS = "1200"

# --- LLM params ---
TEMPERATURE_RERANK = "0.0"
TEMPERATURE_ANSWER = "0.2"
MAX_TOKENS_ANSWER = "900"

# --- Retry params ---
MAX_RETRIES = "3"
RETRY_SLEEP_SECS = "1.0"

# MLflow / UC
# Recomendación: catalog.schema.model
UC_MODEL_NAME = "bind_agent.docs.rag_agent"

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/bind_agent/rag_agent_deploy"

# Serving endpoint para el agente
MODEL_SERVING_ENDPOINT = "bind_agent_rag_agent"


In [0]:
import os, importlib.util
from pathlib import Path

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
# os.environ["DATABRICKS_TOKEN"] = token # Acceso para el service endpoint al search index 
# os.environ["DATABRICKS_HOST"] = host # Acceso para el service endpoint al search index 

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    # "DATABRICKS_TOKEN": os.environ["DATABRICKS_TOKEN"],
    # "DATABRICKS_HOST": os.environ["DATABRICKS_HOST"],
}

In [0]:
# # ==============================
# # 3) Validar que rag_agent.py existe y compila
# # ==============================
# import py_compile
# from pathlib import Path

# agent_py = Path(f"/Workspace/Users/{current_user}/bind_agent/rag/rag_agent.py")
# assert agent_py.exists(), f"No existe: {agent_py}"

# py_compile.compile(str(agent_py), cfile="/tmp/rag_agent.pyc", doraise=True)
# print("✅ rag_agent.py compila OK (pyc en /tmp)")


In [0]:
import shutil
import compileall
import py_compile
from pathlib import Path

ws_base = Path(f"/Workspace/Users/{current_user}/bind_agent/rag/rag_app")
ws_agent = ws_base / "rag_agent.py"
ws_pkg = ws_base / "rag_lib"

assert ws_agent.exists(), f"No existe: {ws_agent}"
assert ws_pkg.exists(), f"No existe: {ws_pkg}"

# Copiar a un filesystem que soporte __pycache__
tmp_base = Path("/tmp/rag_smoketest")
tmp_pkg = tmp_base / "rag_lib"
tmp_agent = tmp_base / "rag_agent.py"

if tmp_base.exists():
    shutil.rmtree(tmp_base)
tmp_base.mkdir(parents=True, exist_ok=True)

shutil.copy2(ws_agent, tmp_agent)
shutil.copytree(ws_pkg, tmp_pkg)

# Compilar en /tmp
ok = compileall.compile_dir(str(tmp_base), force=True, quiet=1)
if not ok:
    raise RuntimeError(f"❌ Falló la compilación en: {tmp_base}")

# (Opcional) también compilar el entrypoint con py_compile
py_compile.compile(str(tmp_agent), cfile="/tmp/rag_agent.pyc", doraise=True)

print("✅ Compilación OK en /tmp:", tmp_base)


In [0]:
# ==============================
# 4) Log + Register (UC) [con signature]
# ==============================
import time
import json
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex

resources = [
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_LLM_ENDPOINT"]),
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_EMBED_ENDPOINT"]),
    DatabricksVectorSearchIndex(index_name=os.environ["RAG_VS_INDEX_FULL_NAME"]),
]

w = WorkspaceClient()
w.workspace.mkdirs(f"/Users/{current_user}/bind_agent")

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.set_registry_uri("databricks-uc")

signature = ModelSignature(
    inputs=Schema([ColSpec("string", "query")]),
    outputs=Schema([
        ColSpec("string", "answer"),
        ColSpec("string", "sources_json"),
        ColSpec("string", "error"),
    ]),
)

class RagAgentUCWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # Esto importa rag_agent.py desde code_paths al cargar el modelo
        import rag_agent
        self._agent = rag_agent.RagAgent()

    def predict(self, context, model_input):
        # Llama al RagAgent real (devuelve list[dict]) y lo normaliza a DataFrame
        out = self._agent.predict(context, model_input)
        rows = []
        for o in out:
            rows.append({
                "answer": o.get("answer", ""),
                "sources_json": json.dumps(o.get("sources", []), ensure_ascii=False),
                "error": o.get("error", ""),
            })
        return pd.DataFrame(rows)

with mlflow.start_run(run_name="rag_agent_deploy") as run:
    logged = mlflow.pyfunc.log_model(
        name="agent",
        python_model=RagAgentUCWrapper(),
        code_paths=[
            str(ws_agent),                              # rag.py
            str(ws_pkg.parent / "rag_lib"),           # carpeta con módulos
        ],
        signature=signature,
        resources=resources,   # ✅ CLAVE
    )
    model_uri = logged.model_uri
    print("Model logged:", model_uri)

    uc_model_info = mlflow.register_model(model_uri=model_uri, name=UC_MODEL_NAME)
    print("Registered in UC:", uc_model_info.name, "version:", uc_model_info.version)

# Esperar READY (registro UC a veces tarda)
client = MlflowClient()
t0 = time.time()
while True:
    mv = client.get_model_version(name=UC_MODEL_NAME, version=uc_model_info.version)
    if mv.status == "READY":
        break
    if time.time() - t0 > 900:
        raise TimeoutError(f"Timeout esperando READY para {UC_MODEL_NAME} v{uc_model_info.version}. Status={mv.status}")
    print("Esperando model READY... status=", mv.status)
    time.sleep(5)

print("✅ Model version READY:", UC_MODEL_NAME, "v", uc_model_info.version)


In [0]:
# ==============================
# 5) Crear/Actualizar Model Serving Endpoint
# ==============================
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
import datetime as dt
from databricks.sdk.errors import NotFound

def ensure_serving_endpoint(endpoint_name, model_name, model_version, workload_size="Small"):
    served = ServedEntityInput(
        name=f"{endpoint_name}-entity",
        entity_name=model_name,
        entity_version=str(model_version),
        workload_size=workload_size,
        scale_to_zero_enabled=True,
        environment_vars=env_vars,
    )

    cfg = EndpointCoreConfigInput(name=endpoint_name, served_entities=[served])

    try:
        w.serving_endpoints.get(endpoint_name)
        print(f"[serving] Endpoint existe: {endpoint_name}")
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served],
            timeout=dt.timedelta(minutes=30),
        )
    except NotFound:
        print(f"[serving] Creando endpoint: {endpoint_name}")
        w.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=cfg,
            timeout=dt.timedelta(minutes=30),
        )

    print("✅ Serving endpoint listo:", endpoint_name)

ensure_serving_endpoint(
    endpoint_name=MODEL_SERVING_ENDPOINT,
    model_name=UC_MODEL_NAME,
    model_version=uc_model_info.version,
    workload_size="Small",
)

In [0]:
# ==============================
# 6) Smoke test (invocar endpoint)
# ==============================
from mlflow.deployments import get_deploy_client
dc = get_deploy_client("databricks")

payload = {
    "dataframe_split": {
        "columns": ["query"],
        "data": [["¿Cuales son las Previsiones de octubre 2025?"]],
    }
}

resp = dc.predict(endpoint=MODEL_SERVING_ENDPOINT, inputs=payload)
print(resp)
